In [1]:
import torch
print(torch.cuda.is_available())   # should be True
print(torch.version.cuda)          # CUDA version PyTorch is built with
print(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else None
print(torch.cuda.device_count())
import os
os.cpu_count()

True
11.7
NVIDIA RTX 6000 Ada Generation
2


48

In [2]:

from typing import Dict, Tuple, List, Optional
import numpy as np
import pandas as pd
import logging
logger = logging.getLogger(__name__)

def load_ap1_data_from_csv(csv_filepath: str, replicate: Optional[int] = None) -> Dict[str, np.ndarray]:
    """
    Loads AP1 single-cell data from CSV or Excel file.

    Args:
        csv_filepath: Path to the CSV or Excel file

    Returns:
        Dictionary with condition identifiers as keys and feature matrices as values
    """
    logger.info(f"Loading data from: {csv_filepath}")

    # Load the data based on file extension
    if csv_filepath.endswith('.csv'):
        df = pd.read_csv(csv_filepath)
    elif csv_filepath.endswith('.xlsx'):
        df = pd.read_excel(csv_filepath)
    else:
        raise ValueError("Unsupported file format. Please provide a .csv or .xlsx file.")
    
    replacement_map = {
                        '0.316 uM Vemurafenib': 'Vem',
                        '0.316 uM Vem + 0.0316 uM Tram': 'Vem+Tram'
                        }
    df['condition'] = df['condition'].replace(replacement_map)

    print(df['condition'].unique())

    # Define AP1 protein features (these are in log space already)
    ap1_features = [
        'cFOS (log a.u.)', 'p-cFOS (log a.u.)', 'FRA1 (log a.u.)', 'p-FRA1 (log a.u.)', 'FRA2 (log a.u.)',
        'cJUN (log a.u.)', 'p-cJUN (log a.u.)', 'JUNB (log a.u.)', 'JUND (log a.u.)', 'p-ATF1 (log a.u.)',
        'ATF2 (log a.u.)',	 'p-ATF2 (log a.u.)', 'ATF3 (log a.u.)', 'ATF4 (log a.u.)', 'p-ATF4 (log a.u.)',
        'ATF5 (log a.u.)', 'ATF6 (log a.u.)', 'MITF (log a.u.)', 'NGFR (log a.u.)', 'p-ERK (log a.u.)',
    ]

    # Check if all features exist
    missing_features = [f for f in ap1_features if f not in df.columns]
    if missing_features:
        logger.warning(f"Missing features: {missing_features}")
        ap1_features = [f for f in ap1_features if f in df.columns]

    logger.info(f"Using {len(ap1_features)} AP1 features")

    # Create condition-based data dictionary
    data_dict = {}

    if replicate is not None:
        # Group by condition, time, and cell line
        for (condition, time, cell_line, replicate_id), group in df.groupby(['condition', 'time', 'cell_line', 'replicate_id']):
            # Create condition identifier
            condition_id = f"{cell_line}_{condition}_{time.replace(' ', '')}_rep{replicate_id}"

            # Extract feature matrix
            feature_matrix = group[ap1_features].values

            # Remove rows with any NaN values
            valid_rows = ~np.isnan(feature_matrix).any(axis=1)
            feature_matrix = feature_matrix[valid_rows]

            if len(feature_matrix) > 0:
                data_dict[condition_id] = feature_matrix
                logger.info(f"Loaded {condition_id}: {feature_matrix.shape}")
            else:
                logger.warning(f"No valid data for {condition_id}")
    else:
        # Group by condition, time, and cell line
        for (condition, time, cell_line), group in df.groupby(['condition', 'time', 'cell_line']):
            # Create condition identifier
            condition_id = f"{cell_line}_{condition}_{time.replace(' ', '')}"

            # Extract feature matrix
            feature_matrix = group[ap1_features].values

            # Remove rows with any NaN values
            valid_rows = ~np.isnan(feature_matrix).any(axis=1)
            feature_matrix = feature_matrix[valid_rows]

            if len(feature_matrix) > 0:
                data_dict[condition_id] = feature_matrix
                logger.info(f"Loaded {condition_id}: {feature_matrix.shape}")
            else:
                logger.warning(f"No valid data for {condition_id}")

    return data_dict

def prepare_pair_from_mat(cell_line: str,
                          baseline_condition: str, baseline_time: str,
                          target_condition: str, target_time: str,
                          replicate: Optional[int] = None) -> Tuple[np.ndarray, np.ndarray]:
    print("Cell line: ", cell_line)
    raw_data_dict = load_ap1_data_from_csv('mmc5.xlsx', replicate)

    if replicate is not None:
        pre_key = f"{cell_line}_{baseline_condition}_{baseline_time}_rep{replicate}"
        post_key = f"{cell_line}_{target_condition}_{target_time}_rep{replicate}"
    else:
        pre_key = f"{cell_line}_{baseline_condition}_{baseline_time}"
        post_key = f"{cell_line}_{target_condition}_{target_time}"

    if pre_key not in raw_data_dict or post_key not in raw_data_dict:
        raise ValueError(f"Pair not found: {pre_key}, {post_key}")

    # Equalize N
    n = min(len(raw_data_dict[pre_key]), len(raw_data_dict[post_key]))
    X_pre_raw = raw_data_dict[pre_key][:n]
    X_post_raw = raw_data_dict[post_key][:n]
    return X_pre_raw, X_post_raw



In [3]:
import os
import sys
import json
import logging
import argparse
import geomloss
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict, Tuple, List, Optional
from umap import UMAP
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import rbf_kernel
from typing import Dict, Tuple, List
from scipy.stats import ks_2samp
from scipy.spatial.distance import cdist
from sklearn.metrics import r2_score

import gc
gc.collect()

def median_heuristic_gamma(X: np.ndarray, Y: np.ndarray) -> float:
    """
    Median heuristic for RBF bandwidth: gamma = 1 / median(||x - y||^2).
    Uses the median of pairwise distances in the pooled set.
    """
    Z = np.vstack([X, Y])
    # Sample if too large for efficiency
    max_samples = 5000
    if Z.shape[0] > max_samples:
        idx = np.random.choice(Z.shape[0], size=max_samples, replace=False)
        Z = Z[idx]
    D2 = cdist(Z, Z, metric='sqeuclidean')
    # Use upper triangular without diagonal
    triu = D2[np.triu_indices_from(D2, k=1)]
    med = np.median(triu[triu > 0]) if np.any(triu > 0) else 1.0
    return 1.0 / max(med, 1e-12)

def mmd_distance(X: np.ndarray, Y: np.ndarray, gamma: float) -> float:
    """
    Unbiased MMD^2 estimator using Gaussian (RBF) kernel, sklearn backend.

    Args:
        X: (n_samples, n_features) first sample
        Y: (m_samples, n_features) second sample
        gamma: RBF kernel bandwidth; if None, uses median heuristic

    Returns:
        Unbiased MMD^2 value
    """
    n = X.shape[0]
    m = Y.shape[0]

    # Kernel matrices
    Kxx = rbf_kernel(X, X, gamma=gamma)
    Kyy = rbf_kernel(Y, Y, gamma=gamma)
    Kxy = rbf_kernel(X, Y, gamma=gamma)

    # Unbiased: exclude diagonal entries
    np.fill_diagonal(Kxx, 0.0)
    np.fill_diagonal(Kyy, 0.0)

    term_xx = Kxx.sum() / (n * (n - 1)) if n > 1 else 0.0
    term_yy = Kyy.sum() / (m * (m - 1)) if m > 1 else 0.0
    term_xy = 2.0 * Kxy.mean()

    mmd2 = term_xx + term_yy - term_xy
    mmd2 = max(mmd2, 0.0)  # Numerical stability
    return float(mmd2)

def r2_feature_means(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """
    R^2 computed across features between mean vectors of y_true and y_pred.
    """
    mu_true = y_true.mean(axis=0)
    mu_pred = y_pred.mean(axis=0)
    ss_res = float(np.sum((mu_true - mu_pred) ** 2))
    ss_tot = float(np.sum((mu_true - mu_true.mean()) ** 2))
    if ss_tot <= 1e-12:
        return 1.0 if ss_res <= 1e-12 else 0.0
    return 1.0 - ss_res / ss_tot

def wasserstein_pointcloud(
    X,
    Y,
    p: int = 2,
    a=None,
    b=None,
    method: str = "emd",          # "emd" (exact) or "sinkhorn" (approx)
    reg: float = 1e-1,            # Sinkhorn regularization (only used if method="sinkhorn")
    return_plan: bool = False,
):
    """
    Compute Wasserstein distance W_p between two empirical distributions supported on point sets X and Y.

    Parameters
    ----------
    X : (n, d) array-like
        Source points.
    Y : (m, d) array-like
        Target points.
    p : int
        Order of the Wasserstein distance (commonly 1 or 2).
    a : (n,) array-like or None
        Weights for X; if None, uniform weights.
    b : (m,) array-like or None
        Weights for Y; if None, uniform weights.
    method : str
        "emd" for exact optimal transport (requires POT),
        "sinkhorn" for entropic approximation (requires POT).
    reg : float
        Entropic regularization strength for Sinkhorn.
    return_plan : bool
        If True, also return the optimal transport plan.

    Returns
    -------
    Wp : float
        Wasserstein distance of order p.
    plan : (n, m) ndarray, optional
        Optimal transport plan (only if return_plan=True).
    """
    X = np.asarray(X, dtype=np.float64)
    Y = np.asarray(Y, dtype=np.float64)
    if X.ndim != 2 or Y.ndim != 2:
        raise ValueError("X and Y must be 2D arrays with shape (n, d) and (m, d).")
    if X.shape[1] != Y.shape[1]:
        raise ValueError(f"Dimension mismatch: X has d={X.shape[1]}, Y has d={Y.shape[1]}.")

    n, d = X.shape
    m, _ = Y.shape

    if a is None:
        a = np.full(n, 1.0 / n, dtype=np.float64)
    else:
        a = np.asarray(a, dtype=np.float64)
        a = a / a.sum()

    if b is None:
        b = np.full(m, 1.0 / m, dtype=np.float64)
    else:
        b = np.asarray(b, dtype=np.float64)
        b = b / b.sum()

    # Cost matrix: C_ij = ||x_i - y_j||^p
    # Compute squared Euclidean via (x-y)^2 = x^2 + y^2 - 2xy for speed
    X2 = np.sum(X * X, axis=1, keepdims=True)          # (n, 1)
    Y2 = np.sum(Y * Y, axis=1, keepdims=True).T        # (1, m)
    sq = np.maximum(X2 + Y2 - 2.0 * (X @ Y.T), 0.0)     # (n, m)
    if p == 2:
        C = sq
    else:
        C = sq ** (p / 2.0)

    try:
        import ot  # POT: Python Optimal Transport
    except ImportError as e:
        raise ImportError(
            "This function requires the POT library. Install with: pip install pot"
        ) from e

    method = method.lower()
    if method == "emd":
        # exact OT: minimizes <P, C>
        P = ot.emd(a, b, C)
        cost = float(np.sum(P * C))
    elif method == "sinkhorn":
        # entropic OT approximation
        P = ot.sinkhorn(a, b, C, reg=reg)
        cost = float(np.sum(P * C))
    else:
        raise ValueError('method must be either "emd" or "sinkhorn".')

    Wp = cost ** (1.0 / p)

    if return_plan:
        return Wp, P
    return Wp

def summarize_metrics(y_true: np.ndarray, y_pred: np.ndarray, median_gamma: float) -> dict:
    """
    Compute a standard set of metrics: MMD^2 (RBF), R^2 of feature means, median KS across features, and Wasserstein distance.
    """
    # Drop any samples that contain NaNs in either true or pred
    mask = (~np.isnan(y_true).any(axis=1)) & (~np.isnan(y_pred).any(axis=1))
    if mask.sum() < len(y_true):
        print(f"[summarize_metrics] Dropping {len(y_true) - mask.sum()} samples with NaNs.")
    
    y_true = y_true[mask]
    y_pred = y_pred[mask]

    out = {}

    out['mmd2_gamma_median'] = mmd_distance(y_true, y_pred, gamma=median_gamma)
    out['mmd2_gamma_0.5'] = mmd_distance(y_true, y_pred, gamma=0.5)
    out['mmd2_gamma_1.0'] = mmd_distance(y_true, y_pred, gamma=1.0)
    out['wasserstein_distance'] = wasserstein_pointcloud(y_true, y_pred, p=2, method="emd")
    out['R2_feature_means'] = r2_feature_means(y_true, y_pred)
    return out

def split_train_test(X: np.ndarray, Y: np.ndarray, train_fraction: float, seed: int = 42) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    if X.shape[0] != Y.shape[0]:
        min_len = min(len(X), len(Y))
        X = X[:min_len]
        Y = Y[:min_len]

    n = X.shape[0]
    n_train = max(1, int(n * train_fraction))
    rng = np.random.default_rng(seed)
    idx = rng.permutation(n)
    tr_idx, te_idx = idx[:n_train], idx[n_train:]
    return X[tr_idx], X[te_idx], Y[tr_idx], Y[te_idx]

def topk_markers(adata, drug: str, k: int = 50, rank_key: str = "marker_genes-drug-rank"):
    R = adata.varm[rank_key]

    # --- get the rank vector for this drug ---
    if hasattr(R, "columns") and hasattr(R, "iloc"):  # pandas DataFrame
        if drug in R.columns:
            r = R[drug].to_numpy()
        else:
            # fallback: interpret columns as ordered groups; try to map via rank_genes_groups names
            names = adata.uns["rank_genes_groups"]["names"]
            groups = list(names.dtype.names) if (hasattr(names, "dtype") and names.dtype.names is not None) else list(names.columns)
            r = R.iloc[:, groups.index(drug)].to_numpy()
    else:  # numpy array (or array-like)
        names = adata.uns["rank_genes_groups"]["names"]
        groups = list(names.dtype.names) if (hasattr(names, "dtype") and names.dtype.names is not None) else list(names.columns)
        r = np.asarray(R)[:, groups.index(drug)]

    # smaller rank => stronger marker
    idx = np.argsort(r)[:k]
    gene_ids = adata.var_names[idx].to_list()
    gene_short = (adata.var.iloc[idx]["gene_short_name"].to_list()
                  if "gene_short_name" in adata.var.columns else None)
    return gene_ids, gene_short, idx


In [4]:
import sys
import torch
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from cellot.models.cellot import load_networks, compute_loss_f, compute_loss_g

from sklearn.metrics.pairwise import rbf_kernel


def mmd_distance(x, y, gamma):
    xx = rbf_kernel(x, x, gamma)
    xy = rbf_kernel(x, y, gamma)
    yy = rbf_kernel(y, y, gamma)

    return xx.mean() + yy.mean() - 2 * xy.mean()

def compute_mmd_loss(lhs, rhs, gammas):
    return np.mean([mmd_distance(lhs, rhs, g) for g in gammas])

from cellot.losses.mmd import mmd_distance

def run_cellot_pair(train_pre: np.ndarray, train_post: np.ndarray,
                    test_pre: np.ndarray, test_post: np.ndarray,
                    layers: Optional[List[int]] = [32, 32 ,32],
                    n_epochs: int = 5000,
                    feature_subset: Optional[List[int]] = None,) -> Dict:
    
    device = 'cuda'
    print(f"VERS torch={torch.__version__} (CellOT), device={device}", file=sys.stderr, flush=True)


    # Apply feature subset if specified
    if feature_subset is not None:
        print(f"Using feature subset of size {len(feature_subset)}", file=sys.stderr, flush=True)
        train_pre = train_pre[:, feature_subset]
        train_post = train_post[:, feature_subset]
        test_pre = test_pre[:, feature_subset]
        test_post = test_post[:, feature_subset]

    # Preprocess: standardize jointly and optionally apply PCA for stability
    X_all = np.vstack([train_pre, train_post])
    scaler = StandardScaler()
    X_all_s = scaler.fit_transform(X_all)
    d = X_all_s.shape[1]
    pca_dims = min(50, d)
    if pca_dims < d:
        pca = PCA(n_components=pca_dims, svd_solver='full', random_state=42)
        X_all_p = pca.fit_transform(X_all_s)
        tr_pre_p = X_all_p[:len(train_pre)]
        tr_post_p = X_all_p[len(train_pre):]
        te_pre_p = pca.transform(scaler.transform(test_pre))
        use_pca = True
    else:
        tr_pre_p = X_all_s[:len(train_pre)]
        tr_post_p = X_all_s[len(train_pre):]
        te_pre_p = scaler.transform(test_pre)
        use_pca = False

    # Networks - Using official CellOT configuration
    input_dim = tr_pre_p.shape[1]
    config = {
        'model': {
            'name': 'cellot',
            'hidden_units': layers,
            'kernel_init_fxn': {'name': 'uniform', 'a': -0.01, 'b': 0.01},
            'activation': 'relu',
            'softplus_W_kernels': True,
            'f': {},
            'g': {}
        }
    }
    f, g = load_networks(config, input_dim=input_dim)
    f = f.to(device).float()
    g = g.to(device).float()

    # Data tensors
    src = torch.tensor(tr_pre_p, dtype=torch.float32, device=device)
    tgt = torch.tensor(tr_post_p, dtype=torch.float32, device=device)
    te_src = torch.tensor(te_pre_p, dtype=torch.float32, device=device)

    # Optimizers matching official config
    lr = 1e-4
    optim_f = torch.optim.Adam(f.parameters(), lr=lr, betas=(0.5, 0.9), weight_decay=0)
    optim_g = torch.optim.Adam(g.parameters(), lr=lr, betas=(0.5, 0.9), weight_decay=0)

    # No schedulers in official config
    # n_epochs = 1200  # More epochs for better convergence
    n_epochs = n_epochs + 1  
    # Training loop following official CellOT implementation
    f.train(); g.train()
    batch_size = 256  # Official config
    n_inner_iters = 10  # Official config


    for epoch in range(n_epochs):
        f.train(); g.train()
        perm_t = torch.randperm(len(tgt), device=device)[:batch_size]
        yt = tgt[perm_t]
        
        # Multiple g updates per iteration (official implementation)
        for _ in range(n_inner_iters):
            perm_s = torch.randperm(len(src), device=device)[:batch_size]
            xs = src[perm_s].detach().clone().requires_grad_(True)
            
            optim_g.zero_grad()
            g_loss = compute_loss_g(f, g, xs).mean()
            g_loss.backward()
            torch.nn.utils.clip_grad_norm_(g.parameters(), max_norm=0.5)
            optim_g.step()
        
        # Single f update (official implementation)
        perm_s = torch.randperm(len(src), device=device)[:batch_size]
        xs = src[perm_s].detach().clone().requires_grad_(True)
        
        optim_f.zero_grad()
        f_loss = compute_loss_f(f, g, xs, yt).mean()
        f_loss.backward()
        optim_f.step()
        
        # Clamp weights for f (official implementation)
        if hasattr(f, 'clamp_w'):
            f.clamp_w()
        
        
        # ---- Evaluate train MMD and early-stop ----
        if epoch % 50 == 0: 
            f.eval()
            g.eval()


            # Transport a fixed subset of training PRE (in preprocessed space)
            tr_src_eval = src.requires_grad_(True)
            tr_pred_p = g.transport(tr_src_eval).detach().cpu().numpy()
            # Invert preprocessing to original space (so MMD is comparable to your final eval)
            if use_pca:
                tr_pred = scaler.inverse_transform(pca.inverse_transform(tr_pred_p))
            else:
                tr_pred = scaler.inverse_transform(tr_pred_p)
            train_mmd_min = mmd_distance(train_post, tr_pred, gamma=1.0)


            te_src_full = te_src.detach().clone().requires_grad_(True)
            te_pred_full = g.transport(te_src_full).detach().cpu().numpy()
            if use_pca:
                te_pred_inv_full = scaler.inverse_transform(pca.inverse_transform(te_pred_full))
            else:
                te_pred_inv_full = scaler.inverse_transform(te_pred_full)
            test_metrics = mmd_distance(test_post, te_pred_inv_full, gamma=median_gamma)

            print(
                f"[CellOT] epoch={epoch} f_loss={f_loss.item():.4f} g_loss={g_loss.item():.4f} | "
                f"train mmd={train_mmd_min:.4f} | "
                f"test_mmd={test_metrics:.4f}",
                file=sys.stderr,
                flush=True,
            )

                
            

    # Inference (CellOT transport requires gradients for autodiff)
    f.eval(); g.eval()
    # CellOT needs gradients even in eval mode for transport computation
    te_src_for_transport = te_src.detach().clone().requires_grad_(True)
    te_tx = g.transport(te_src_for_transport).detach().cpu().numpy()

    # Inverse preprocess
    if use_pca:
        te_tx_inv = scaler.inverse_transform(pca.inverse_transform(te_tx))
    else:
        te_tx_inv = scaler.inverse_transform(te_tx)
    # Final evaluation
    metrics = summarize_metrics(test_post[:len(te_tx_inv)], te_tx_inv, median_gamma)

    gammas = np.logspace(1, -3, num=50)
    mmd = compute_mmd_loss(test_post[:len(te_tx_inv)], te_tx_inv, gammas=gammas)
    print(f"[CellOT] Final CellOT MMD: {mmd:.4f}", file=sys.stderr, flush=True)
    
    return {'y_pred': te_tx_inv, 'metrics': metrics}
    


In [5]:
drug = "Vem"
X_pre, X_post = prepare_pair_from_mat('COLO858', 'DMSO','24h', drug, '72h')
jfe_indices = [1, 6, 0, 5, 4, 7, 8, 2, 3, 19]  

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre, X_post, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)

# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre[:, jfe_indices], Y_tr_post[:, jfe_indices], X_te_pre[:, jfe_indices], Y_te_post[:, jfe_indices], n_epochs=2000)
    metrics = summarize_metrics(out["y_pred"], Y_te_post[:, jfe_indices], median_gamma)
    print(f"Run {run} metrics: {metrics}")
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))


from umap import UMAP
import matplotlib.pyplot as plt

source = Y_tr_post[:, jfe_indices]
target = Y_te_post[:, jfe_indices]
predicted = out.get('y_pred') 

# Instantiate UMAP
umap_model = UMAP(n_components=2, random_state=42)

all_sample_umap = umap_model.fit_transform(np.vstack([source, target]))
source_umap = umap_model.transform(source)
target_umap = umap_model.transform(target)
y_pred_umap = umap_model.transform(predicted)

fig, ax = plt.subplots(figsize=(4, 4))
# ax.scatter(source_umap[:, 0], source_umap[:, 1], s=10, alpha=0.7, label='train_post', color='C2')
ax.scatter(target_umap[:, 0], target_umap[:, 1], s=10, alpha=0.7, label='observed treated cells', color="#C88131")
ax.scatter(y_pred_umap[:, 0], y_pred_umap[:, 1], s=10, alpha=0.7, label='predicted cells', color="#1F4D8D")

ax.set_title(f'{drug}')
# ax.set_xlabel('UMAP 1')
# ax.set_ylabel('UMAP 2')
ax.set_aspect('equal', 'box')
# Add a legend to distinguish the points
ax.legend()
# Adjust layout
plt.tight_layout()
# Display the plot
plt.savefig(f"./plots/cellot_on_4i_drug_{drug}.png", dpi=300)

Cell line:  COLO858
['DMSO' 'Vem' 'Vem+Tram']
X_pre cells: (3026, 20)
X_post cells: (3026, 20)
(2420, 20)
(606, 20)
(2420, 20)
(606, 20)


VERS torch=1.13.1+cu117 (CellOT), device=cuda


Median heuristic gamma: 0.05162262759745905
**************** Run: 0 ****************


[CellOT] epoch=0 f_loss=-1.3571 g_loss=-5.0167 | train mmd=0.2892 | test_mmd=1.3294
[CellOT] epoch=50 f_loss=-2.5143 g_loss=0.3889 | train mmd=0.5300 | test_mmd=0.3511
[CellOT] epoch=100 f_loss=-2.7093 g_loss=3.3984 | train mmd=0.7644 | test_mmd=0.3322
[CellOT] epoch=150 f_loss=-4.6768 g_loss=5.7451 | train mmd=0.7639 | test_mmd=0.2845
[CellOT] epoch=200 f_loss=-6.2082 g_loss=8.3595 | train mmd=0.7725 | test_mmd=0.2526
[CellOT] epoch=250 f_loss=-8.3589 g_loss=10.3283 | train mmd=0.7793 | test_mmd=0.2200
[CellOT] epoch=300 f_loss=-9.2954 g_loss=12.8297 | train mmd=0.7534 | test_mmd=0.1925
[CellOT] epoch=350 f_loss=-10.7652 g_loss=15.3954 | train mmd=0.7190 | test_mmd=0.1685
[CellOT] epoch=400 f_loss=-11.8829 g_loss=16.8155 | train mmd=0.7151 | test_mmd=0.1418
[CellOT] epoch=450 f_loss=-12.4099 g_loss=18.5495 | train mmd=0.7033 | test_mmd=0.1227
[CellOT] epoch=500 f_loss=-12.7849 g_loss=21.1363 | train mmd=0.6769 | test_mmd=0.1006
[CellOT] epoch=550 f_loss=-14.6765 g_loss=22.5376 | train

Run 0 metrics: {'mmd2_gamma_median': 0.0008898991539725287, 'mmd2_gamma_0.5': 0.007069895455005026, 'mmd2_gamma_1.0': 0.010359813910979315, 'wasserstein_distance': 0.8182263571257119, 'R2_feature_means': 0.9980937226362605}
**************** Run: 1 ****************


...

[CellOT] epoch=50 f_loss=-2.2892 g_loss=0.5087 | train mmd=0.5296 | test_mmd=0.3615
[CellOT] epoch=100 f_loss=-2.9831 g_loss=3.4274 | train mmd=0.7483 | test_mmd=0.3242
[CellOT] epoch=150 f_loss=-4.8987 g_loss=6.4117 | train mmd=0.7726 | test_mmd=0.2932
[CellOT] epoch=200 f_loss=-6.8844 g_loss=9.1379 | train mmd=0.7726 | test_mmd=0.2531
[CellOT] epoch=250 f_loss=-8.5420 g_loss=10.5204 | train mmd=0.7214 | test_mmd=0.2149
[CellOT] epoch=300 f_loss=-10.3428 g_loss=14.1010 | train mmd=0.7738 | test_mmd=0.1981
[CellOT] epoch=350 f_loss=-10.6274 g_loss=14.6018 | train mmd=0.7213 | test_mmd=0.1618
[CellOT] epoch=400 f_loss=-11.7444 g_loss=18.5489 | train mmd=0.6985 | test_mmd=0.1371
[CellOT] epoch=450 f_loss=-13.3784 g_loss=19.5322 | train mmd=0.6761 | test_mmd=0.1154
[CellOT] epoch=500 f_loss=-11.1356 g_loss=20.1556 | train mmd=0.6215 | test_mmd=0.0934
[CellOT] epoch=550 f_loss=-11.2872 g_loss=23.6440 | train mmd=0.6053 | test_mmd=0.0775
[CellOT] epoch=600 f_loss=-12.9464 g_loss=24.8391 | t

Run 8 metrics: {'mmd2_gamma_median': 0.0022774238370297795, 'mmd2_gamma_0.5': 0.01424091953093476, 'mmd2_gamma_1.0': 0.0190023251082215, 'wasserstein_distance': 0.8918732790423525, 'R2_feature_means': 0.9942331961038458}
**************** Run: 9 ****************


[CellOT] epoch=50 f_loss=-2.2321 g_loss=0.8794 | train mmd=0.5110 | test_mmd=0.3691
[CellOT] epoch=100 f_loss=-2.9668 g_loss=3.8101 | train mmd=0.7691 | test_mmd=0.3355
[CellOT] epoch=150 f_loss=-4.9126 g_loss=5.9245 | train mmd=0.7562 | test_mmd=0.2899
[CellOT] epoch=200 f_loss=-6.8247 g_loss=8.3989 | train mmd=0.7290 | test_mmd=0.2534
[CellOT] epoch=250 f_loss=-7.9488 g_loss=10.5668 | train mmd=0.7516 | test_mmd=0.2269
[CellOT] epoch=300 f_loss=-9.9818 g_loss=12.3380 | train mmd=0.7548 | test_mmd=0.2001
[CellOT] epoch=350 f_loss=-10.3673 g_loss=15.1801 | train mmd=0.6951 | test_mmd=0.1682
[CellOT] epoch=400 f_loss=-11.2173 g_loss=17.0635 | train mmd=0.7196 | test_mmd=0.1504
[CellOT] epoch=450 f_loss=-12.3667 g_loss=18.3096 | train mmd=0.6771 | test_mmd=0.1260
[CellOT] epoch=500 f_loss=-12.0766 g_loss=20.3262 | train mmd=0.6309 | test_mmd=0.1017
[CellOT] epoch=550 f_loss=-13.3576 g_loss=23.7932 | train mmd=0.6053 | test_mmd=0.0832
[CellOT] epoch=600 f_loss=-13.9132 g_loss=25.1361 | tr

Run 9 metrics: {'mmd2_gamma_median': 0.001978776996309106, 'mmd2_gamma_0.5': 0.03915400267308511, 'mmd2_gamma_1.0': 0.05330024544615042, 'wasserstein_distance': 0.9318318257802892, 'R2_feature_means': 0.9983560635644584}
                        mean     std
mmd2_gamma_median     0.0019  0.0007
mmd2_gamma_0.5        0.0220  0.0106
mmd2_gamma_1.0        0.0301  0.0142
wasserstein_distance  0.8919  0.0458
R2_feature_means      0.9965  0.0019


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


In [6]:
drug = "Vem"
X_pre, X_post = prepare_pair_from_mat('WM902B', 'DMSO','24h', drug, '72h')
jfe_indices = [1, 6, 0, 5, 4, 7, 8, 2, 3, 19]  

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre, X_post, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)

# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre[:, jfe_indices], Y_tr_post[:, jfe_indices], X_te_pre[:, jfe_indices], Y_te_post[:, jfe_indices], n_epochs=2000)
    metrics = summarize_metrics(out["y_pred"], Y_te_post[:, jfe_indices], median_gamma)
    print(f"Run {run} metrics: {metrics}")
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))


from umap import UMAP
import matplotlib.pyplot as plt

source = Y_tr_post[:, jfe_indices]
target = Y_te_post[:, jfe_indices]
predicted = out.get('y_pred') 

# Instantiate UMAP
umap_model = UMAP(n_components=2, random_state=42)

all_sample_umap = umap_model.fit_transform(np.vstack([source, target]))
source_umap = umap_model.transform(source)
target_umap = umap_model.transform(target)
y_pred_umap = umap_model.transform(predicted)

fig, ax = plt.subplots(figsize=(4, 4))
# ax.scatter(source_umap[:, 0], source_umap[:, 1], s=10, alpha=0.7, label='train_post', color='C2')
ax.scatter(target_umap[:, 0], target_umap[:, 1], s=10, alpha=0.7, label='observed treated cells', color="#C88131")
ax.scatter(y_pred_umap[:, 0], y_pred_umap[:, 1], s=10, alpha=0.7, label='predicted cells', color="#1F4D8D")

ax.set_title(f'{drug}')
# ax.set_xlabel('UMAP 1')
# ax.set_ylabel('UMAP 2')
ax.set_aspect('equal', 'box')
# Add a legend to distinguish the points
ax.legend()
# Adjust layout
plt.tight_layout()
# Display the plot
plt.savefig(f"./plots/cellot_on_4i_drug_{drug}.png", dpi=300)

Cell line:  WM902B
['DMSO' 'Vem' 'Vem+Tram']
X_pre cells: (5690, 20)
X_post cells: (5690, 20)
(4552, 20)
(1138, 20)
(4552, 20)
(1138, 20)


VERS torch=1.13.1+cu117 (CellOT), device=cuda


Median heuristic gamma: 0.06456030933401641
**************** Run: 0 ****************


[CellOT] epoch=0 f_loss=-1.3478 g_loss=-4.2979 | train mmd=0.4075 | test_mmd=1.3562
[CellOT] epoch=50 f_loss=-2.0507 g_loss=0.4978 | train mmd=0.5396 | test_mmd=0.2764
[CellOT] epoch=100 f_loss=-2.4600 g_loss=2.1598 | train mmd=0.6309 | test_mmd=0.2202
[CellOT] epoch=150 f_loss=-4.0853 g_loss=4.2834 | train mmd=0.6577 | test_mmd=0.1989
[CellOT] epoch=200 f_loss=-5.6215 g_loss=5.8955 | train mmd=0.6331 | test_mmd=0.1690
[CellOT] epoch=250 f_loss=-7.3941 g_loss=7.7300 | train mmd=0.6004 | test_mmd=0.1391
[CellOT] epoch=300 f_loss=-7.2905 g_loss=8.9752 | train mmd=0.5553 | test_mmd=0.1113
[CellOT] epoch=350 f_loss=-7.7490 g_loss=11.1106 | train mmd=0.5165 | test_mmd=0.0891
[CellOT] epoch=400 f_loss=-8.3747 g_loss=11.5345 | train mmd=0.4686 | test_mmd=0.0674
[CellOT] epoch=450 f_loss=-9.3412 g_loss=13.4609 | train mmd=0.4196 | test_mmd=0.0495
[CellOT] epoch=500 f_loss=-8.7416 g_loss=14.0152 | train mmd=0.3637 | test_mmd=0.0343
[CellOT] epoch=550 f_loss=-7.7250 g_loss=14.2650 | train mmd=0.

Run 0 metrics: {'mmd2_gamma_median': 0.0012542330548115377, 'mmd2_gamma_0.5': 0.004492773892949664, 'mmd2_gamma_1.0': 0.005741941641809523, 'wasserstein_distance': 0.6023647707603519, 'R2_feature_means': 0.9984231791671069}
**************** Run: 1 ****************


...

[CellOT] epoch=0 f_loss=1.5957 g_loss=-4.4153 | train mmd=0.4190 | test_mmd=1.1334
[CellOT] epoch=50 f_loss=-2.0146 g_loss=0.1903 | train mmd=0.4758 | test_mmd=0.2579
[CellOT] epoch=100 f_loss=-2.9091 g_loss=2.1354 | train mmd=0.6733 | test_mmd=0.2311
[CellOT] epoch=150 f_loss=-4.4628 g_loss=5.4215 | train mmd=0.7008 | test_mmd=0.2100
[CellOT] epoch=200 f_loss=-6.4984 g_loss=7.0216 | train mmd=0.6532 | test_mmd=0.1708
[CellOT] epoch=250 f_loss=-7.3479 g_loss=8.4152 | train mmd=0.6286 | test_mmd=0.1434
[CellOT] epoch=300 f_loss=-8.3354 g_loss=11.2761 | train mmd=0.5880 | test_mmd=0.1161
[CellOT] epoch=350 f_loss=-9.4553 g_loss=11.7949 | train mmd=0.5367 | test_mmd=0.0926
[CellOT] epoch=400 f_loss=-10.1883 g_loss=12.9789 | train mmd=0.5073 | test_mmd=0.0754
[CellOT] epoch=450 f_loss=-10.4643 g_loss=15.3211 | train mmd=0.4760 | test_mmd=0.0592
[CellOT] epoch=500 f_loss=-9.4750 g_loss=15.6545 | train mmd=0.4064 | test_mmd=0.0407
[CellOT] epoch=550 f_loss=-8.6773 g_loss=16.3277 | train mmd=

Run 8 metrics: {'mmd2_gamma_median': 0.0018728046209228744, 'mmd2_gamma_0.5': 0.005815766138460021, 'mmd2_gamma_1.0': 0.006482459634980231, 'wasserstein_distance': 0.609433181302826, 'R2_feature_means': 0.9975236818763463}
**************** Run: 9 ****************


[CellOT] epoch=0 f_loss=-0.7550 g_loss=-6.5383 | train mmd=0.3823 | test_mmd=1.4637
[CellOT] epoch=50 f_loss=-1.3337 g_loss=0.3096 | train mmd=0.5685 | test_mmd=0.2876
[CellOT] epoch=100 f_loss=-2.8117 g_loss=2.1314 | train mmd=0.6677 | test_mmd=0.2347
[CellOT] epoch=150 f_loss=-4.6845 g_loss=4.5533 | train mmd=0.6743 | test_mmd=0.2071
[CellOT] epoch=200 f_loss=-5.7380 g_loss=6.6481 | train mmd=0.6443 | test_mmd=0.1705
[CellOT] epoch=250 f_loss=-7.6351 g_loss=8.0465 | train mmd=0.5957 | test_mmd=0.1389
[CellOT] epoch=300 f_loss=-7.8210 g_loss=10.1745 | train mmd=0.5498 | test_mmd=0.1134
[CellOT] epoch=350 f_loss=-9.1816 g_loss=11.5537 | train mmd=0.5206 | test_mmd=0.0929
[CellOT] epoch=400 f_loss=-9.8632 g_loss=11.4419 | train mmd=0.4831 | test_mmd=0.0729
[CellOT] epoch=450 f_loss=-10.2393 g_loss=12.9908 | train mmd=0.4373 | test_mmd=0.0538
[CellOT] epoch=500 f_loss=-10.4462 g_loss=14.4519 | train mmd=0.4229 | test_mmd=0.0447
[CellOT] epoch=550 f_loss=-10.0927 g_loss=14.9911 | train mm

Run 9 metrics: {'mmd2_gamma_median': 0.0009923370626707673, 'mmd2_gamma_0.5': 0.004976473842692064, 'mmd2_gamma_1.0': 0.00565701149007547, 'wasserstein_distance': 0.5983066609104358, 'R2_feature_means': 0.9995492797218528}
                        mean     std
mmd2_gamma_median     0.0015  0.0006
mmd2_gamma_0.5        0.0052  0.0010
mmd2_gamma_1.0        0.0061  0.0008
wasserstein_distance  0.6090  0.0083
R2_feature_means      0.9982  0.0010


RuntimeError: can't start new thread

In [7]:
drug = "Vem"
X_pre, X_post = prepare_pair_from_mat('SKMEL19', 'DMSO','24h', drug, '72h')
jfe_indices = [1, 6, 0, 5, 4, 7, 8, 2, 3, 19]  

print("X_pre cells:", X_pre.shape)
print("X_post cells:", X_post.shape)

X_tr_pre, X_te_pre, Y_tr_post, Y_te_post = split_train_test(X_pre, X_post, 0.8)

print(X_tr_pre.shape)
print(X_te_pre.shape)
print(Y_tr_post.shape)
print(Y_te_post.shape)

# Compute median heuristic gamma on training data
median_gamma = median_heuristic_gamma(X_tr_pre, Y_tr_post)
print("Median heuristic gamma:", median_gamma)


all_metrics = []
for run in range(10):
    print(f"**************** Run: {run} ****************")
    seed = 1234 + run
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    out = run_cellot_pair(X_tr_pre[:, jfe_indices], Y_tr_post[:, jfe_indices], X_te_pre[:, jfe_indices], Y_te_post[:, jfe_indices], n_epochs=2000)
    metrics = summarize_metrics(out["y_pred"], Y_te_post[:, jfe_indices], median_gamma)
    print(f"Run {run} metrics: {metrics}")
    all_metrics.append(metrics)

# Results summary
df = pd.DataFrame(all_metrics)
print(df.describe().T[['mean', 'std']].round(4))


from umap import UMAP
import matplotlib.pyplot as plt

source = Y_tr_post[:, jfe_indices]
target = Y_te_post[:, jfe_indices]
predicted = out.get('y_pred') 

# Instantiate UMAP
umap_model = UMAP(n_components=2, random_state=42)

all_sample_umap = umap_model.fit_transform(np.vstack([source, target]))
source_umap = umap_model.transform(source)
target_umap = umap_model.transform(target)
y_pred_umap = umap_model.transform(predicted)

fig, ax = plt.subplots(figsize=(4, 4))
# ax.scatter(source_umap[:, 0], source_umap[:, 1], s=10, alpha=0.7, label='train_post', color='C2')
ax.scatter(target_umap[:, 0], target_umap[:, 1], s=10, alpha=0.7, label='observed treated cells', color="#C88131")
ax.scatter(y_pred_umap[:, 0], y_pred_umap[:, 1], s=10, alpha=0.7, label='predicted cells', color="#1F4D8D")

ax.set_title(f'{drug}')
# ax.set_xlabel('UMAP 1')
# ax.set_ylabel('UMAP 2')
ax.set_aspect('equal', 'box')
# Add a legend to distinguish the points
ax.legend()
# Adjust layout
plt.tight_layout()
# Display the plot
plt.savefig(f"./plots/cellot_on_4i_drug_{drug}.png", dpi=300)

Cell line:  SKMEL19
['DMSO' 'Vem' 'Vem+Tram']
X_pre cells: (2677, 20)
X_post cells: (2677, 20)
(2141, 20)
(536, 20)
(2141, 20)
(536, 20)


VERS torch=1.13.1+cu117 (CellOT), device=cuda


Median heuristic gamma: 0.10487158680337198
**************** Run: 0 ****************


[CellOT] epoch=0 f_loss=-0.3313 g_loss=-5.6712 | train mmd=0.3145 | test_mmd=0.8853
[CellOT] epoch=50 f_loss=-2.3354 g_loss=-1.9843 | train mmd=0.1017 | test_mmd=0.0865
[CellOT] epoch=100 f_loss=-0.4245 g_loss=-0.0805 | train mmd=0.0851 | test_mmd=0.0218
[CellOT] epoch=150 f_loss=-0.0761 g_loss=0.6526 | train mmd=0.0387 | test_mmd=0.0036
[CellOT] epoch=200 f_loss=-0.1050 g_loss=0.9193 | train mmd=0.0165 | test_mmd=0.0034
[CellOT] epoch=250 f_loss=-0.5806 g_loss=1.3326 | train mmd=0.0056 | test_mmd=0.0013
[CellOT] epoch=300 f_loss=0.3165 g_loss=1.2259 | train mmd=0.0040 | test_mmd=0.0016
[CellOT] epoch=350 f_loss=-0.3509 g_loss=1.4420 | train mmd=0.0044 | test_mmd=0.0019
[CellOT] epoch=400 f_loss=0.2738 g_loss=1.4090 | train mmd=0.0035 | test_mmd=0.0026
[CellOT] epoch=450 f_loss=-0.1944 g_loss=1.7903 | train mmd=0.0042 | test_mmd=0.0023
[CellOT] epoch=500 f_loss=0.0196 g_loss=1.7564 | train mmd=0.0026 | test_mmd=0.0008
[CellOT] epoch=550 f_loss=0.0408 g_loss=1.5857 | train mmd=0.0026 | 

Run 0 metrics: {'mmd2_gamma_median': 0.001320746864281297, 'mmd2_gamma_0.5': 0.004396997576266215, 'mmd2_gamma_1.0': 0.005598257084086622, 'wasserstein_distance': 0.6342140424279016, 'R2_feature_means': 0.998732808192168}
**************** Run: 1 ****************


...

[CellOT] epoch=50 f_loss=-1.4129 g_loss=-1.3289 | train mmd=0.1173 | test_mmd=0.0829
[CellOT] epoch=100 f_loss=-1.3042 g_loss=0.0087 | train mmd=0.2079 | test_mmd=0.0446
[CellOT] epoch=150 f_loss=-0.9382 g_loss=1.4222 | train mmd=0.1638 | test_mmd=0.0305
[CellOT] epoch=200 f_loss=-1.4448 g_loss=2.6212 | train mmd=0.1441 | test_mmd=0.0195
[CellOT] epoch=250 f_loss=-1.7412 g_loss=3.0482 | train mmd=0.1152 | test_mmd=0.0105
[CellOT] epoch=300 f_loss=-0.9326 g_loss=2.7844 | train mmd=0.0891 | test_mmd=0.0058
[CellOT] epoch=350 f_loss=-1.7178 g_loss=3.8652 | train mmd=0.0702 | test_mmd=0.0039
[CellOT] epoch=400 f_loss=-1.3702 g_loss=4.3389 | train mmd=0.0770 | test_mmd=0.0027
[CellOT] epoch=450 f_loss=0.5533 g_loss=4.8365 | train mmd=0.0829 | test_mmd=0.0016
[CellOT] epoch=500 f_loss=-0.2493 g_loss=4.2586 | train mmd=0.0578 | test_mmd=0.0012
[CellOT] epoch=550 f_loss=0.5643 g_loss=5.2537 | train mmd=0.0383 | test_mmd=0.0019
[CellOT] epoch=600 f_loss=0.6415 g_loss=7.1521 | train mmd=0.0265 |

Run 8 metrics: {'mmd2_gamma_median': 0.0012576819660510274, 'mmd2_gamma_0.5': 0.00492906910710611, 'mmd2_gamma_1.0': 0.00659249380527982, 'wasserstein_distance': 0.6438661963048388, 'R2_feature_means': 0.9979908670946189}
**************** Run: 9 ****************


[CellOT] epoch=0 f_loss=-0.5349 g_loss=-4.7464 | train mmd=0.2847 | test_mmd=0.9832
[CellOT] epoch=50 f_loss=-1.4623 g_loss=-1.7422 | train mmd=0.1334 | test_mmd=0.1042
[CellOT] epoch=100 f_loss=-0.7991 g_loss=0.3912 | train mmd=0.0897 | test_mmd=0.0223
[CellOT] epoch=150 f_loss=-0.7542 g_loss=1.0103 | train mmd=0.1015 | test_mmd=0.0152
[CellOT] epoch=200 f_loss=-0.6188 g_loss=1.3966 | train mmd=0.1034 | test_mmd=0.0105
[CellOT] epoch=250 f_loss=-0.8771 g_loss=2.6092 | train mmd=0.0765 | test_mmd=0.0057
[CellOT] epoch=300 f_loss=-0.4810 g_loss=2.4090 | train mmd=0.0541 | test_mmd=0.0033
[CellOT] epoch=350 f_loss=-0.1741 g_loss=2.2137 | train mmd=0.0481 | test_mmd=0.0015
[CellOT] epoch=400 f_loss=-0.1135 g_loss=3.1462 | train mmd=0.0295 | test_mmd=0.0016
[CellOT] epoch=450 f_loss=0.5215 g_loss=3.2123 | train mmd=0.0294 | test_mmd=0.0011
[CellOT] epoch=500 f_loss=0.8677 g_loss=3.1728 | train mmd=0.0263 | test_mmd=0.0025
[CellOT] epoch=550 f_loss=-0.5122 g_loss=4.2341 | train mmd=0.0069 |

Run 9 metrics: {'mmd2_gamma_median': 0.002116835808132045, 'mmd2_gamma_0.5': 0.006504946011012813, 'mmd2_gamma_1.0': 0.0076680169659653075, 'wasserstein_distance': 0.6421072001703861, 'R2_feature_means': 0.9982041719924265}
                        mean     std
mmd2_gamma_median     0.0016  0.0004
mmd2_gamma_0.5        0.0057  0.0011
mmd2_gamma_1.0        0.0073  0.0012
wasserstein_distance  0.6417  0.0069
R2_feature_means      0.9980  0.0007


/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/u/jrp5td/here/miniconda3/envs/scgen-env/lib/python3.9/site-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
